# Delta-CeNN V5.5 — Optimized 24-layer Qwen3.5 Experiment

**Evidence-driven upgrade of V5.4.** The prior run installed 18 gated-delta CeNN layers, rejected all six full-attention replacements, achieved 98.52% diagnostic teacher agreement but 0.143× prefill speed and had a tokenizer chat bug. This notebook tests a **chunkwise Qwen gated-delta backend with trainable CeNN state coupling** to avoid Python token loops, retains validation-gated six full-attention V5.3 candidates, repairs fast native/student generation, and exports reproducible artifacts.

**Scientific distinction:** V5.5 is *not* an independent gated-delta replacement: it reuses the original Qwen chunkwise gated-delta recurrence and modifies persistent state between chunks through CeNN diffusion. Only the trained cellular coupling is newly substituted. Neither speed nor quality is assumed until tested on GPU. No input upload is required.

## 0 · GPU environment and source validation
Set **Runtime → Change runtime type → GPU**, then **Run all**. The notebook clones canonical GitHub source, verifies Python/adapter tests and records the repository/model revision. The public Qwen model downloads without an HF token; write access is only needed for optional Hugging Face publishing at the end.

In [ ]:
%pip -q install "transformers==5.17.0" "huggingface_hub>=0.36,<2" "datasets>=3,<5" "accelerate>=1" "safetensors" "pandas>=2" "matplotlib>=3.7" "pytest>=8"


In [ ]:
import os,sys,subprocess,gc,json,hashlib,math,random,time,csv,zipfile,traceback,shutil
from pathlib import Path
from contextlib import contextmanager,nullcontext
import numpy as np,pandas as pd,torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
REPO=Path('/content/TinyCeNN-LM')
if not (REPO/'.git').is_dir():
    subprocess.run(['git','clone','--quiet','--depth','1','--branch','main',
        'https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','--quiet','origin','main'],check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'],check=True)
sys.path.insert(0,str(REPO/'src'))
from tinycenn_lm.cenn_v53_multilayer import (FULL_ATTENTION_LAYERS,FullAttentionGraftManager,
    validate_full_attention_targets,check_quality,experiment_fingerprint,append_jsonl)
from tinycenn_lm.delta_cenn_qwen35 import Qwen35DeltaCeNNLinear
from tinycenn_lm.delta_cenn_qwen35_v55 import Qwen35DeltaCeNNChunkLinear
from tinycenn_lm.delta_cenn_v55_io import write_adapter_package,prepare_hf_model_card
from tinycenn_lm.cenn_deltacell_v3 import CeNNDeltaCellV3
from tinycenn_lm.cenn_head_rescue_v5 import Qwen35HeadRescueV5
from tinycenn_lm.cenn_head_rescue_v51 import CeNNQueryRescueV51,warmstart_v5_delta_core
from tinycenn_lm.cenn_head_rescue_v52 import CeNNHeadRescueV52,warmstart_v51_to_v52
from tinycenn_lm.cenn_head_rescue_v53 import CeNNHeadRescueV53,warmstart_v52_to_v53
from transformers import AutoConfig,AutoTokenizer,Qwen3_5ForCausalLM
from huggingface_hub import model_info
subprocess.run([sys.executable,'-m','pytest','-q',
    str(REPO/'tests/test_delta_cenn_qwen35.py'),
    str(REPO/'tests/test_cenn_head_rescue_v53.py'),
    str(REPO/'tests/test_qwen35_v53_all24_contract.py'),
    str(REPO/'tests/test_delta_cenn_v54_io.py'),
    str(REPO/'tests/test_delta_cenn_v55_chunk.py'),
    str(REPO/'tests/test_qwen35_v55_notebook.py')],check=True)
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → NVIDIA GPU'
DEVICE=torch.device('cuda')
DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
GIT_SHA=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
SOURCE_FILES=[REPO/'src/tinycenn_lm'/f for f in ('delta_cenn_qwen35.py',
    'delta_cenn_v55_io.py','delta_cenn_qwen35_v55.py','cenn_v53_multilayer.py','cenn_deltacell_v3.py','cenn_head_rescue_v5.py',
    'cenn_head_rescue_v51.py','cenn_head_rescue_v52.py','cenn_head_rescue_v53.py')]
SOURCE_SHA256=hashlib.sha256(b''.join(p.read_bytes() for p in SOURCE_FILES)).hexdigest()
print('Git:',GIT_SHA,'model dtype:',DTYPE,'device:',torch.cuda.get_device_name(0))


## 1 · V5.5 chunk-cell experiment

Defaults use **two cellular state updates per 128-token example**, vs V5.4's FP32 Python loop across all tokens. If starting weights already match the native layer closely (validation cosine ≥0.9999, NMSE ≤0.0001), skip unnecessary optimizer steps while retaining a step-zero checkpoint. Full-attention layers are still trained but only installed after their cumulative quality gate passes. `HF_UPLOAD_MODE='quality_only'` is deliberately conservative.

In [ ]:
# @title 24-layer Delta-CeNN / CeNN V5.3 research controls
PROFILE='quick' # @param ['quick','research']
FORCE_ALL_24=False # @param {type:'boolean'}
REPLACEMENT_MODE='one_group' # @param ['one_group','both_groups']
AUTO_DOWNLOAD_RESULTS=True # @param {type:'boolean'}
MODEL_ID='Qwen/Qwen3.5-0.8B'
SEQ_LEN=128 if PROFILE=='quick' else 256
TRAIN_BLOCKS=48 if PROFILE=='quick' else 160
DEV_BLOCKS=24 if PROFILE=='quick' else 64
AUDIT_BLOCKS=16 if PROFILE=='quick' else 48
AUDIT_START_TOKEN=1_850_000
BATCH_SIZE=1 if PROFILE=='quick' else 2
DELTA_STEPS=80 if PROFILE=='quick' else 320
BASELINE_STEPS=60 if PROFILE=='quick' else 240
V53_STEPS=96 if PROFILE=='quick' else 400
DELTA_LR=5e-5
BASELINE_LR=2e-3
V53_LR=6e-4
DELTA_FEATURE_DIM=None  # None = native full-rank key features; compressed < key dimension is experimental
DELTA_CELL_INTERVAL=64   # Two cellular state updates per 128-token sequence
BACKEND_CHUNK_SIZE=64  # Qwen chunked gated-delta primitive; not independent recurrence
SKIP_NEAR_EXACT_FIT=True # Skip training when step-zero already has excellent local parity
FEATURE_DIM=64
LOCAL_WINDOW=64 if PROFILE=='quick' else 128
CHUNK_SIZE=32
RANK=16
SEED=5318
LOG_EVERY=10 if PROFILE=='quick' else 25
MIN_COSINE=.95;MIN_AGREEMENT=.98;MAX_DELTA_NLL=.10;MIN_SPEEDUP=1.05
EARLY_STOP_PATIENCE=5 if PROFILE=='quick' else 9
EARLY_STOP_TOL=1e-5
RUN_FAST_EVAL=True  # True: real-token NLL, next-token gold accuracy and greedy text probes
MAX_NEW_TOKENS=8 if PROFILE=='quick' else 24
HF_UPLOAD_MODE='quality_only'  # @param ['research','quality_only','disabled']
HF_REPO_ID='vtava/Qwen35-08B-DeltaCeNN-V55-ChunkCell-Research'
ENABLE_COLAB_SECRET_LOOKUP=False  # Avoid HF_TOKEN vault timeout; set env HF_TOKEN or enable explicitly
OUT=Path('/content/qwen35_delta_cenn_all24_results')
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'checkpoints').mkdir(exist_ok=True)
TRAIN_EVENTS=OUT/'training_steps.jsonl'
TEXT_LOG=OUT/'training.log'
STAGE_CSV=OUT/'cumulative_layer_results.csv'
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32=True
print('Full-attention mode',REPLACEMENT_MODE,'force all 24',FORCE_ALL_24)
print('Optimizers: Delta',DELTA_STEPS,'full seed',BASELINE_STEPS,'V5.3',V53_STEPS,'HF mode',HF_UPLOAD_MODE)


## 2 · Load and inspect the original Qwen3.5 hybrid backbone

Verify **six** full-attention layers (SDPA, special V5.3 wrapper) and **18** gated-delta layers (`linear_attn`, Delta-CeNN wrapper). Pretrained projection and MLP weights remain frozen. Validate wrapper numerical parity before substituting any layer.

In [ ]:
REVISION=model_info(MODEL_ID).sha
cfg=AutoConfig.from_pretrained(MODEL_ID,revision=REVISION).get_text_config(decoder=True)
assert cfg.model_type=='qwen3_5_text' and cfg.hidden_size==1024 and cfg.num_hidden_layers==24
FULL_TARGETS=tuple(i for i,t in enumerate(cfg.layer_types) if t=='full_attention')
DELTA_TARGETS=tuple(i for i,t in enumerate(cfg.layer_types) if t=='linear_attention')
assert FULL_TARGETS==FULL_ATTENTION_LAYERS and len(DELTA_TARGETS)==18
TARGET_LAYERS=tuple(range(cfg.num_hidden_layers))
assert sorted(FULL_TARGETS+DELTA_TARGETS)==list(TARGET_LAYERS)
TOKENIZER=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,use_fast=True)
model=Qwen3_5ForCausalLM.from_pretrained(MODEL_ID,revision=REVISION,dtype=DTYPE,
    attn_implementation='sdpa',low_cpu_mem_usage=True).to(DEVICE).eval()
model.requires_grad_(False);model.config.use_cache=False
LAYERS=model.model.layers
MANAGER=FullAttentionGraftManager(LAYERS,FULL_TARGETS)
DELTA_NATIVE={i:LAYERS[i].linear_attn for i in DELTA_TARGETS}
INSTALLED_DELTA={}
HD=int(MANAGER.native[11].head_dim)
QHEADS=int(MANAGER.native[11].q_proj.out_features//(2*HD))
KVHEADS=int(MANAGER.native[11].k_proj.out_features//HD)
assert (QHEADS,KVHEADS,HD)==(8,2,256)
EXACT_GROUPS=(0,) if REPLACEMENT_MODE=='one_group' else ()
APPROX_KV=KVHEADS-len(EXACT_GROUPS)
APPROX_Q=APPROX_KV*(QHEADS//KVHEADS)
@contextmanager
def native_everywhere():
    restored={i:LAYERS[i].linear_attn for i in DELTA_TARGETS}
    with MANAGER.native_context():
        try:
            for i,original in DELTA_NATIVE.items():LAYERS[i].linear_attn=original
            yield
        finally:
            for i,old in restored.items():LAYERS[i].linear_attn=old
@contextmanager
def delta_candidate(idx,adapter):
    assert idx in DELTA_NATIVE and adapter.original is DELTA_NATIVE[idx]
    assert LAYERS[idx].linear_attn is DELTA_NATIVE[idx]
    LAYERS[idx].linear_attn=adapter
    try:yield
    finally:LAYERS[idx].linear_attn=DELTA_NATIVE[idx]
def install_delta(idx,adapter):
    assert idx in DELTA_NATIVE and adapter.original is DELTA_NATIVE[idx]
    assert LAYERS[idx].linear_attn is DELTA_NATIVE[idx]
    LAYERS[idx].linear_attn=adapter
    INSTALLED_DELTA[idx]=adapter
def installed_indices():
    return sorted(set(MANAGER.installed_layers)|set(INSTALLED_DELTA))
# Numerical parity controls on a text-only prefix, before training.
probe=TOKENIZER('Delta CeNN compares causal learned memory with original Qwen.',
    add_special_tokens=False,return_tensors='pt').input_ids.to(DEVICE)
with torch.inference_mode(),torch.autocast('cuda',dtype=DTYPE):
    original_logits=model(input_ids=probe,use_cache=False,logits_to_keep=0).logits.float()
    oracle=Qwen35HeadRescueV5(MANAGER.native[FULL_TARGETS[0]],None,exact_groups=tuple(range(KVHEADS))).eval()
    with MANAGER.candidate(FULL_TARGETS[0],oracle):
        exact_logits=model(input_ids=probe,use_cache=False,logits_to_keep=0).logits.float()
full_parity=float((original_logits-exact_logits).abs().max())
assert full_parity<.05 and bool((original_logits.argmax(-1)==exact_logits.argmax(-1)).all())
linear_idx=DELTA_TARGETS[0]
# Two DISTINCT controls. The native-backend oracle proves that convolution,
# QKV projections, gates, normalization, and output projection are wired
# correctly. The genuine Delta-CeNN tokenwise recurrence is then measured as
# an experimental approximation, not mislabeled a bit-exact chunk kernel.
backend_oracle=Qwen35DeltaCeNNLinear(DELTA_NATIVE[linear_idx],feature_dim=None,
    cells=False,native_backend_oracle=True).to(DEVICE).eval()
token_oracle=Qwen35DeltaCeNNLinear(DELTA_NATIVE[linear_idx],feature_dim=None,
    cells=False).to(DEVICE).eval()
with torch.inference_mode(),torch.autocast('cuda',dtype=DTYPE):
    with delta_candidate(linear_idx,backend_oracle):
        backend_logits=model(input_ids=probe,use_cache=False,logits_to_keep=0).logits.float()
    with delta_candidate(linear_idx,token_oracle):
        token_logits=model(input_ids=probe,use_cache=False,logits_to_keep=0).logits.float()

def parity_metrics(reference, candidate):
    a,b=reference.float(),candidate.float()
    return dict(max_abs=float((a-b).abs().max().item()),
        nmse=float(((a-b).square().mean()/a.square().mean().clamp_min(1e-12)).item()),
        cosine=float(F.cosine_similarity(a.reshape(-1,a.shape[-1]),
                   b.reshape(-1,b.shape[-1]),dim=-1).mean().item()),
        top1_agreement=float((a.argmax(-1)==b.argmax(-1)).float().mean().item()))
PARITY=dict(layer=int(linear_idx),backend='Qwen3.5 chunk gated-delta',
    model_revision=REVISION,source_git_sha=GIT_SHA,dtype=str(DTYPE),
    exact_full_attention=parity_metrics(original_logits,exact_logits),
    adapter_backend_oracle=parity_metrics(original_logits,backend_logits),
    experimental_token_delta=parity_metrics(original_logits,token_logits))
(OUT/'parity_diagnostics.json').write_text(json.dumps(PARITY,indent=2,allow_nan=False))
print('PARITY CONTROLS',json.dumps(PARITY,indent=2))
# Keep a STRICT backend parity requirement: if this fails, the projection or
# backend integration is actually wrong. Do NOT relax the threshold based on
# the experimental tokenwise recurrence's much larger absolute logit errors.
assert PARITY['adapter_backend_oracle']['max_abs']<.05, (
    'Native-backend adapter integration failed: inspect parity_diagnostics.json')
assert PARITY['adapter_backend_oracle']['top1_agreement']==1.0, (
    'Native-backend adapter changed teacher top-1; inspect parity diagnostics')
if PARITY['experimental_token_delta']['max_abs']>=.15:
    print('NOTE: tokenwise Delta-CeNN diverges numerically from the chunk backend. '
          'This is measured approximation error, not native-adapter parity failure; '
          'continue only as an experimental student and require strict validation gates.')
print('Pinned model',REVISION,'full layers',FULL_TARGETS,'delta layers',DELTA_TARGETS)
del oracle,backend_oracle,token_oracle,probe,original_logits,exact_logits,backend_logits,token_logits
# Diagnostic output is already saved, even if the subsequent training fails.
torch.cuda.empty_cache()


## 2A · Native vs V5.4 tokenwise vs V5.5 chunkwise speed screen

This controlled GPU sanity benchmark uses the **same frozen Qwen input and one gated-delta layer**. It measures native, prior V5.4 tokenwise recurrence, and new V5.5 chunk-cell recurrence before any training. Only the V5.5 branch performs additional cellular diffusion. These figures do **not** establish all-24-layer speedup; the later paired audit measures that separately.

In [ ]:
# @title Single-layer engine comparison on actual Qwen text
@torch.inference_mode()
def benchmark_recurrence_backends():
    sample=TOKENIZER(('Comparing fast recurrent memory updates against the native '
                      'Qwen gated-delta model. ')*9,add_special_tokens=False,
                     return_tensors='pt')['input_ids'][:,:96].to(DEVICE)
    idx=DELTA_TARGETS[0]
    modes={
      'native':None,
      'v54_tokenwise':Qwen35DeltaCeNNLinear(DELTA_NATIVE[idx],feature_dim=None,
                      cells=True,cell_interval=16).to(DEVICE).eval(),
      'v55_chunk_cell':Qwen35DeltaCeNNChunkLinear(DELTA_NATIVE[idx],feature_dim=None,
                      cells=True,cell_interval=DELTA_CELL_INTERVAL,
                      backend_chunk_size=BACKEND_CHUNK_SIZE).to(DEVICE).eval(),
    }
    benchmark={}
    for name,adapter in modes.items():
        readings=[]
        context=(delta_candidate(idx,adapter) if adapter is not None else nullcontext())
        with context:
            for attempt in range(9):
                begin=torch.cuda.Event(enable_timing=True)
                end=torch.cuda.Event(enable_timing=True)
                begin.record()
                with torch.autocast('cuda',dtype=DTYPE):
                    logits=model(input_ids=sample,use_cache=False,
                                 logits_to_keep=1).logits
                end.record();torch.cuda.synchronize()
                assert bool(torch.isfinite(logits).all())
                if attempt>=3:readings.append(begin.elapsed_time(end))
        benchmark[name]=dict(ms_median=float(np.median(readings)),
                             ms_iqr=float(np.percentile(readings,75)-np.percentile(readings,25)))
        print(name,benchmark[name],flush=True)
    benchmark['v55_vs_tokenwise_speedup']=(benchmark['v54_tokenwise']['ms_median']/
                                           benchmark['v55_chunk_cell']['ms_median'])
    benchmark['v55_vs_native_speedup']=(benchmark['native']['ms_median']/
                                        benchmark['v55_chunk_cell']['ms_median'])
    (OUT/'engine_benchmark.json').write_text(json.dumps(benchmark,indent=2))
    return benchmark
ENGINE_BENCHMARK=benchmark_recurrence_backends()


## 3 · Data provenance and frozen native-teacher baseline

Train on WikiText-2 train prefix, rank and gate on validation, then audit once on a predeclared disjoint portion of the train partition (not an independent blinded benchmark). Prior test-split results are not reused. All 24 stages share a single immutable training/validation data selection.

In [ ]:
from datasets import load_dataset
def token_windows(split,count,length,start_token=0):
    ds=load_dataset('Salesforce/wikitext','wikitext-2-raw-v1',split=split)
    needed=start_token+count*length
    ids=[]
    for row in ds:
        sentence=row['text'].strip()
        if not sentence:continue
        ids.extend(TOKENIZER.encode(sentence,add_special_tokens=False))
        ids.append(TOKENIZER.eos_token_id)
        if len(ids)>=needed:break
    if len(ids)<needed:raise RuntimeError(f'{split} has {len(ids)} tokens; need {needed}')
    return torch.tensor(ids[start_token:needed],dtype=torch.long).reshape(count,length)
TRAIN=token_windows('train',TRAIN_BLOCKS,SEQ_LEN)
DEV=token_windows('validation',DEV_BLOCKS,SEQ_LEN)
DATA_HASHES={name:hashlib.sha256(data.numpy().tobytes()).hexdigest()
    for name,data in (('train',TRAIN),('validation',DEV))}
assert TRAIN_BLOCKS*SEQ_LEN<AUDIT_START_TOKEN
CONFIG=dict(model=MODEL_ID,revision=REVISION,repo_sha=GIT_SHA,source_sha256=SOURCE_SHA256,
    profile=PROFILE,force_all24=FORCE_ALL_24,replacement_mode=REPLACEMENT_MODE,
    all_targets=TARGET_LAYERS,full_targets=FULL_TARGETS,delta_targets=DELTA_TARGETS,
    seq_len=SEQ_LEN,train_blocks=TRAIN_BLOCKS,dev_blocks=DEV_BLOCKS,
    audit_blocks=AUDIT_BLOCKS,audit_offset=AUDIT_START_TOKEN,
    delta_steps=DELTA_STEPS,baseline_steps=BASELINE_STEPS,v53_steps=V53_STEPS,
    delta_feature_dim=DELTA_FEATURE_DIM,delta_cell_interval=DELTA_CELL_INTERVAL,
    batch_size=BATCH_SIZE,local_window=LOCAL_WINDOW,seed=SEED,data_hashes=DATA_HASHES,
    torch_version=torch.__version__,gpu=torch.cuda.get_device_name(0))
RUN_FINGERPRINT=experiment_fingerprint(CONFIG)
(OUT/'config.json').write_text(json.dumps(CONFIG,indent=2,allow_nan=False))
print('Train',TRAIN.shape,'Validation',DEV.shape,'fingerprint',RUN_FINGERPRINT[:16])


## 4 · Sequential teacher capture, full-model metrics, training logs and paired latency

Each target layer is calibrated using the **actual upstream prefix already installed**. Distinct capture interfaces preserve full-attention RoPE and gated-delta tensor signatures. Every training step goes to `training_steps.jsonl`, and each stage is logged to `training.log`. Native timing temporarily restores **both** full-attention and linear-attention modules.

In [ ]:
def log_line(message):
    line=f'[{time.strftime("%Y-%m-%dT%H:%M:%S")}] {message}'
    print(line,flush=True)
    with TEXT_LOG.open('a',encoding='utf-8') as f:f.write(line+'\n')
@torch.inference_mode()
def inference(blocks,keep=1):
    with torch.autocast('cuda',dtype=DTYPE):
        return model(input_ids=blocks.to(DEVICE),use_cache=False,logits_to_keep=keep)
@torch.inference_mode()
def capture_target(idx,split,blocks):
    full=idx in FULL_TARGETS
    native=MANAGER.native[idx] if full else DELTA_NATIVE[idx]
    assert (LAYERS[idx].self_attn if full else LAYERS[idx].linear_attn) is native
    stored={k:[] for k in (('x','y','cos','sin') if full else ('x','y'))}
    def hook(mod,args,kwargs,out):
        x=kwargs.get('hidden_states')
        if x is None and args:x=args[0]
        y=out[0] if isinstance(out,tuple) else out
        stored['x'].append(x.detach().to('cpu',dtype=torch.float16))
        stored['y'].append(y.detach().to('cpu',dtype=torch.float16))
        if full:
            cos,sin=kwargs['position_embeddings']
            stored['cos'].append(cos.detach().cpu().float())
            stored['sin'].append(sin.detach().cpu().float())
    handle=native.register_forward_hook(hook,with_kwargs=True)
    try:
        for start in range(0,len(blocks),BATCH_SIZE):inference(blocks[start:start+BATCH_SIZE])
    finally:handle.remove()
    answer={key:torch.cat(values,0).contiguous() for key,values in stored.items()}
    assert answer['x'].shape[:2]==blocks.shape
    return answer
def captured_batch(data,indices):
    x=data['x'][indices].to(DEVICE,dtype=DTYPE)
    y=data['y'][indices].to(DEVICE,dtype=DTYPE)
    rope=(data['cos'][indices].to(DEVICE),data['sin'][indices].to(DEVICE)) if 'cos' in data else None
    return x,y,rope
def forward_adapter(adapter,x,rope):
    if isinstance(adapter,Qwen35DeltaCeNNLinear):return adapter(x)
    output,_=adapter(x,position_embeddings=rope)
    return output
def mixer_loss(pred,teacher):
    p=pred.float();t=teacher.float()
    nmse=(p-t).square().mean()/t.square().mean().clamp_min(1e-8)
    return nmse+.2*(1-F.cosine_similarity(p,t,dim=-1).mean())
@torch.inference_mode()
def local_validation(adapter,data):
    adapter.eval();err=energy=cosine_sum=0.;n=0
    for lo in range(0,len(data['x']),BATCH_SIZE):
        indices=torch.arange(lo,min(lo+BATCH_SIZE,len(data['x'])))
        x,y,rope=captured_batch(data,indices)
        with torch.autocast('cuda',dtype=DTYPE):pred=forward_adapter(adapter,x,rope)
        p,t=pred.float(),y.float()
        err+=(p-t).square().sum().item();energy+=t.square().sum().item()
        cosine_sum+=F.cosine_similarity(p,t,dim=-1).sum().item()
        n+=p.shape[0]*p.shape[1]
    return dict(nmse=err/max(energy,1e-8),cosine=cosine_sum/max(n,1))
@torch.inference_mode()
def model_metrics(blocks,teacher_top1=None):
    n=0;nll=0.;correct=0;agreement=0;tops=[]
    for lo in range(0,len(blocks),BATCH_SIZE):
        block=blocks[lo:lo+BATCH_SIZE]
        positions=torch.arange(block.shape[1]-1,device=DEVICE)
        logits=inference(block,keep=positions).logits.float()
        gold=block[:,1:].to(DEVICE)
        logp=F.log_softmax(logits,dim=-1)
        nll-=logp.gather(-1,gold.unsqueeze(-1)).sum().item()
        pred=logits.argmax(-1)
        correct+=(pred==gold).sum().item();n+=gold.numel()
        if teacher_top1 is None:tops.append(pred.cpu().to(torch.int32))
        else:
            reference=teacher_top1[lo//BATCH_SIZE].to(DEVICE)
            assert reference.shape==pred.shape
            agreement+=(reference==pred).sum().item()
    out=dict(nll=nll/n,accuracy=correct/n,decisions=n)
    if teacher_top1 is not None:out['teacher_agreement']=agreement/n
    return out,tops
@torch.inference_mode()
def paired_prefill(blocks,warmups=2,repeats=6):
    sample=blocks[:min(BATCH_SIZE,len(blocks))].to(DEVICE)
    native_times=[];student_times=[]
    def one(native):
        with native_everywhere() if native else nullcontext():
            begin=torch.cuda.Event(enable_timing=True);end=torch.cuda.Event(enable_timing=True)
            begin.record();out=inference(sample,keep=1).logits
            end.record();torch.cuda.synchronize()
            assert bool(torch.isfinite(out).all())
            return begin.elapsed_time(end)
    for i in range(warmups+repeats):
        if i%2:student=one(False);native=one(True)
        else:native=one(True);student=one(False)
        if i>=warmups:native_times.append(native);student_times.append(student)
    return dict(native_ms=float(np.median(native_times)),
        student_ms=float(np.median(student_times)),
        speedup=float(np.median(native_times)/np.median(student_times)),
        native_iqr_ms=float(np.percentile(native_times,75)-np.percentile(native_times,25)),
        student_iqr_ms=float(np.percentile(student_times,75)-np.percentile(student_times,25)),
        repeats=repeats)


## 5 · V5.5 chunked Delta-CeNN + V5.3 full attention

Linear layers use a **chunked Qwen gated-delta primitive**, interleaved with learned CeNN spatial diffusion of the persistent key-memory cells. Original Qwen gates, convolution and output projection remain frozen. The six full-attention layers still use the distinct V5.3 head-rescue branch, trained and screened independently. Failed full-attention candidates are saved under `candidates/`, not installed.

Do not interpret the chunk primitive as a wholly novel CeNN recurrence. Benchmark native, V5.4 tokenwise and V5.5 chunked versions separately.

In [ ]:
def make_base(idx):
    core=CeNNDeltaCellV3(APPROX_Q,APPROX_KV,HD,
        feature_dim=FEATURE_DIM,chunk_size=CHUNK_SIZE,mode='none',
        local_window=LOCAL_WINDOW).to(DEVICE)
    return Qwen35HeadRescueV5(MANAGER.native[idx],core,exact_groups=EXACT_GROUPS).eval()
def make_v53(idx,base_state):
    common=dict(num_heads=APPROX_Q,num_kv_heads=APPROX_KV,head_dim=HD,
        feature_dim=FEATURE_DIM,chunk_size=CHUNK_SIZE,local_window=LOCAL_WINDOW,
        correction_rank=RANK,query_router=True,query_correction=True)
    v51=CeNNQueryRescueV51(**common);warmstart_v5_delta_core(v51,base_state)
    v52=CeNNHeadRescueV52(**common,post_rank=RANK,post_correction=True)
    warmstart_v51_to_v52(v52,v51.state_dict())
    v53=CeNNHeadRescueV53(**common,post_rank=RANK,post_correction=True,
        cross_rank=RANK,cross_correction=True,output_calibration=True)
    warmstart_v52_to_v53(v53,v52.state_dict())
    return Qwen35HeadRescueV5(MANAGER.native[idx],v53.to(DEVICE),exact_groups=EXACT_GROUPS).eval()
def make_delta(idx):
    return Qwen35DeltaCeNNChunkLinear(DELTA_NATIVE[idx],feature_dim=DELTA_FEATURE_DIM,
        cells=True,cell_interval=DELTA_CELL_INTERVAL,
        backend_chunk_size=BACKEND_CHUNK_SIZE).to(DEVICE).eval()
def fit_local(idx,stage,adapter,train,dev,steps,lr):
    """Best-of-start + train with validation-based patience; preserve source checkpoint."""
    core=adapter.core
    core.requires_grad_(True)
    optimizer=torch.optim.AdamW(core.parameters(),lr=lr,weight_decay=.005)
    schedule=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=steps,eta_min=lr*.1)
    scaler=torch.amp.GradScaler('cuda',enabled=(DTYPE==torch.float16))
    generator=torch.Generator().manual_seed(SEED+idx*29+(0 if stage=='delta_cenn' else 5000))
    indices=torch.randint(0,len(train['x']),(steps,BATCH_SIZE),generator=generator)
    initial=local_validation(adapter,dev)
    best=initial['nmse']+.2*(1-initial['cosine'])
    best_local=initial
    best_sd={k:v.detach().cpu().clone() for k,v in core.state_dict().items()}
    best_step=0;stale=0;performed=0;began=time.perf_counter()
    append_jsonl(TRAIN_EVENTS,dict(layer=idx,stage=stage,step=0,
        val_nmse=initial['nmse'],val_cosine=initial['cosine'],
        note='frozen step-zero candidate before learning'))
    log_line(f'L{idx} {stage} init cosine={initial["cosine"]:.6f} NMSE={initial["nmse"]:.6f}')
    for step in range(1,steps+1):
        ix=indices[step-1];x,y,rope=captured_batch(train,ix)
        adapter.train();optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=DTYPE):
            pred=forward_adapter(adapter,x,rope)
            loss=mixer_loss(pred,y)
        if not bool(torch.isfinite(loss)):
            raise FloatingPointError(f'L{idx}/{stage} step {step}: non-finite loss')
        scaler.scale(loss).backward();scaler.unscale_(optimizer)
        grad=torch.nn.utils.clip_grad_norm_(core.parameters(),.5)
        scaler.step(optimizer);scaler.update();schedule.step()
        performed=step
        event=dict(layer=idx,family='linear_attention' if idx in DELTA_TARGETS else 'full_attention',
            stage=stage,step=step,total_steps=steps,loss=float(loss.detach()),
            grad_norm=float(grad),lr=float(optimizer.param_groups[0]['lr']),
            elapsed_seconds=float(time.perf_counter()-began))
        if step==1 or step==steps or step%LOG_EVERY==0:
            stats=local_validation(adapter,dev)
            score=stats['nmse']+.2*(1-stats['cosine'])
            if score+EARLY_STOP_TOL<best:
                best=score;best_local=stats;best_step=step;stale=0
                best_sd={k:v.detach().cpu().clone() for k,v in core.state_dict().items()}
            else:
                stale+=1
            event.update(val_cosine=stats['cosine'],val_nmse=stats['nmse'],
                         best_step=best_step,best_score=best)
            log_line(f'L{idx} {stage} {step}/{steps} cos={stats["cosine"]:.5f} '
                f'best={best_local["cosine"]:.5f} step={best_step}')
            if stale>=EARLY_STOP_PATIENCE:
                event['early_stop']=True
                append_jsonl(TRAIN_EVENTS,event)
                log_line(f'L{idx} {stage}: early stopping step {step}; best step {best_step}')
                break
        append_jsonl(TRAIN_EVENTS,event)
    core.load_state_dict(best_sd,strict=True);core.requires_grad_(False);adapter.eval()
    STAGE_TRAINING[(idx,stage)]=dict(requested_steps=steps,performed_steps=performed,
        best_step=best_step,early_stopped=performed<steps,initial_validation=initial,
        best_validation=best_local)
    return best_sd,best_local
@torch.inference_mode()
def warmstart_parity(base,updated,dev):
    x,_,rope=captured_batch(dev,torch.tensor([0]))
    with torch.autocast('cuda',dtype=DTYPE):
        a=forward_adapter(base,x,rope);b=forward_adapter(updated,x,rope)
    diff=float((a-b).abs().max())
    assert diff<.01,diff
    return diff


## 6 · Progressive training and explicit quality gating
Train and evaluate each candidate against native Qwen. With `FORCE_ALL_24=False`, **failed candidates are saved but not installed**, so damage cannot silently accumulate. The diagnostic audit is deferred until after selection. Complete and partial checkpoint files, full training steps and logs are retained.

In [ ]:
STAGE_TRAINING={}
REPORT=[];RUN_OK=False;RUN_ERROR=None;RUN_START=time.time()
NATIVE_DEV=None;FINAL=None;ARCHIVE=None
try:
    log_line('START Delta-CeNN + CeNN V5.3 / all 24 '+RUN_FINGERPRINT[:16])
    with native_everywhere():NATIVE_DEV,NATIVE_TOP1=model_metrics(DEV)
    log_line('NATIVE validation '+json.dumps(NATIVE_DEV))
    for layer in TARGET_LAYERS:
        family='full_attention' if layer in FULL_TARGETS else 'linear_attention'
        log_line(f'BEGIN layer {layer} {family}; installed prefix {installed_indices()}')
        train_capture=capture_target(layer,'train',TRAIN)
        dev_capture=capture_target(layer,'dev',DEV)
        torch.manual_seed(SEED+layer*13)
        warmstart_diff=0.0
        if family=='linear_attention':
            adapter=make_delta(layer)
            starting=local_validation(adapter,dev_capture)
            if (SKIP_NEAR_EXACT_FIT and starting['cosine']>=0.9999 and
                    starting['nmse']<=1e-4):
                # Preserve neutral initial CeNN coupling rather than spend GPU
                # time fitting a nearly exact chunk recurrence (V5.4: best_step=0).
                state={k:v.detach().cpu().clone() for k,v in adapter.core.state_dict().items()}
                local=starting
                STAGE_TRAINING[(layer,'delta_cenn')]=dict(
                    requested_steps=DELTA_STEPS,performed_steps=0,best_step=0,
                    early_stopped=True,reason='step_zero_parity_screen',
                    initial_validation=starting,best_validation=starting)
                log_line(f'L{layer} chunk-cell step-zero accepted for screening: '+str(starting))
            else:
                state,local=fit_local(layer,'delta_cenn',adapter,
                    train_capture,dev_capture,DELTA_STEPS,DELTA_LR)
            steps=dict(delta_cenn=DELTA_STEPS)
        else:
            baseline=make_base(layer)
            base_state,_=fit_local(layer,'full_delta_seed',baseline,
                train_capture,dev_capture,BASELINE_STEPS,BASELINE_LR)
            torch.save(dict(core_state=base_state,layer=layer,stage='full_delta_seed',
                fingerprint=RUN_FINGERPRINT),OUT/'checkpoints'/f'L{layer:02d}_full_seed.pt')
            adapter=make_v53(layer,base_state)
            warmstart_diff=warmstart_parity(baseline,adapter,dev_capture)
            del baseline
            torch.cuda.empty_cache()
            state,local=fit_local(layer,'full_v53',adapter,
                train_capture,dev_capture,V53_STEPS,V53_LR)
            steps=dict(full_delta_seed=BASELINE_STEPS,full_v53=V53_STEPS)
        checkpoint=OUT/'checkpoints'/f'L{layer:02d}_{family}_core.pt'
        torch.save(dict(core_state=state,layer=layer,family=family,
            core_config=adapter.core.config if hasattr(adapter.core,'config') else dict(
                feature_dim=adapter.core.feature_dim,cell_interval=adapter.core.cell_interval),
            validation=local,fingerprint=RUN_FINGERPRINT,model_revision=REVISION,
            source_git_sha=GIT_SHA,replacement_mode=REPLACEMENT_MODE),checkpoint)
        context=MANAGER.candidate(layer,adapter) if family=='full_attention' else delta_candidate(layer,adapter)
        with context:
            student_dev,_=model_metrics(DEV,teacher_top1=NATIVE_TOP1)
            # Timing within the candidate context includes this layer plus installed prefix.
            timing=paired_prefill(DEV,warmups=2,repeats=4 if PROFILE=='quick' else 12)
        delta_nll=student_dev['nll']-NATIVE_DEV['nll']
        gate=check_quality(local['cosine'],student_dev['teacher_agreement'],delta_nll,
            timing['speedup'],min_cosine=MIN_COSINE,min_agreement=MIN_AGREEMENT,
            max_delta_nll=MAX_DELTA_NLL,min_speedup=MIN_SPEEDUP)
        install=bool(FORCE_ALL_24 or gate.quality_pass)
        if not install:
            log_line(f'REJECT L{layer}: cosine={local["cosine"]:.5f}, '
                f'agreement={student_dev["teacher_agreement"]:.5f}, '
                f'delta_NLL={delta_nll:+.5f}; preserving native layer')
        if install:
            if family=='full_attention':MANAGER.install(layer,adapter)
            else:install_delta(layer,adapter)
        row=dict(layer=layer,family=family,installed=install,
            forced=bool(install and not gate.quality_pass),
            installed_count=len(installed_indices()),
            local_cosine=local['cosine'],local_nmse=local['nmse'],
            cumulative_dev_nll=student_dev['nll'],dev_delta_nll=delta_nll,
            cumulative_dev_agreement=student_dev['teacher_agreement'],
            cumulative_dev_accuracy=student_dev['accuracy'],
            native_ms=timing['native_ms'],student_ms=timing['student_ms'],
            cumulative_speedup=timing['speedup'],quality_pass=gate.quality_pass,
            deploy_pass=gate.deploy_pass,warmstart_max_abs_diff=warmstart_diff,
            trainable_core_params=sum(p.numel() for p in adapter.core.parameters()),
            training_meta={key:STAGE_TRAINING[(layer,key)] for key in steps},**steps)
        REPORT.append(row);pd.DataFrame(REPORT).to_csv(STAGE_CSV,index=False)
        log_line('LAYER SUMMARY '+json.dumps(row,allow_nan=False))
        del state,train_capture,dev_capture
        gc.collect();torch.cuda.empty_cache()
    AUDIT=token_windows('train',AUDIT_BLOCKS,SEQ_LEN,start_token=AUDIT_START_TOKEN)
    DATA_HASHES['diagnostic_train_tail']=hashlib.sha256(AUDIT.numpy().tobytes()).hexdigest()
    assert len(set(DATA_HASHES.values()))==3
    with native_everywhere():NATIVE_AUDIT,AUDIT_TOP1=model_metrics(AUDIT)
    STUDENT_AUDIT,_=model_metrics(AUDIT,teacher_top1=AUDIT_TOP1)
    FINAL_TIMING=paired_prefill(AUDIT,warmups=4,repeats=16)
    all24=len(installed_indices())==24
    stage_quality=all(row['quality_pass'] for row in REPORT if row['installed'])
    audit_delta=STUDENT_AUDIT['nll']-NATIVE_AUDIT['nll']
    FINAL_PASS=bool(all24 and stage_quality and audit_delta<=MAX_DELTA_NLL
                    and STUDENT_AUDIT['teacher_agreement']>=MIN_AGREEMENT
                    and FINAL_TIMING['speedup']>MIN_SPEEDUP)
    FINAL=dict(architecture='Qwen3.5-0.8B: 18 chunk-cell Delta-CeNN + six V5.3 head rescue',
       parity_controls=PARITY,
       model_revision=REVISION,source_git_sha=GIT_SHA,source_sha256=SOURCE_SHA256,
       installed=installed_indices(),full_attention_targets=FULL_TARGETS,
       gated_delta_targets=DELTA_TARGETS,forced_all_24=FORCE_ALL_24,
       all24_installed=all24,all_stage_quality_pass=stage_quality,final_pass=FINAL_PASS,
       native_validation=NATIVE_DEV,stage_results=REPORT,native_audit=NATIVE_AUDIT,
       student_audit=STUDENT_AUDIT,audit_delta_nll=audit_delta,
       paired_audit=FINAL_TIMING,audit_source='disjoint WikiText-2 train tail, not external test',
       data_hashes=DATA_HASHES,run_fingerprint=RUN_FINGERPRINT,
       runtime_seconds=time.time()-RUN_START,
       limitations='V5.5 reuses Qwen chunked recurrence with learned CeNN state coupling (not independent recurrence); non-fused FP32 cell updates; text-only prefill; no Qwen hybrid KV cache or generation; forced mode is diagnostic.')
    (OUT/'final_audit.json').write_text(json.dumps(FINAL,indent=2,allow_nan=False))
    log_line('FINAL AUDIT '+json.dumps({k:FINAL[k] for k in (
        'all24_installed','all_stage_quality_pass','final_pass','audit_delta_nll','paired_audit')},allow_nan=False))
    if not FINAL_PASS:log_line('REJECT for deployment: research 24-layer artifacts saved, native Qwen remains the reference.')
    RUN_OK=True
except BaseException as exc:
    RUN_ERROR=repr(exc)
    log_line('EXPERIMENT FAILED (preserving completed layers): '+RUN_ERROR)
    with (OUT/'error_traceback.txt').open('w') as f:traceback.print_exc(file=f)
    raise
finally:
    pd.DataFrame(REPORT).to_csv(STAGE_CSV,index=False)
    status=dict(completed=RUN_OK,error=RUN_ERROR,installed=installed_indices(),
        run_fingerprint=RUN_FINGERPRINT,runtime_seconds=time.time()-RUN_START)
    (OUT/'run_status.json').write_text(json.dumps(status,indent=2,allow_nan=False))
    if REPORT:
        try:
            f=pd.DataFrame(REPORT);fig,ax=plt.subplots(figsize=(11,4))
            ax.plot(f.layer,f.cumulative_dev_agreement,marker='.',label='Teacher agreement')
            ax.axhline(MIN_AGREEMENT,ls='--',color='gray')
            ax.set(xlabel='Decoder layer replaced',ylabel='Native top-1 agreement',
                title='Cumulative 24-layer Delta-CeNN + V5.3 experiment')
            ax.legend();fig.tight_layout();fig.savefig(OUT/'cumulative_agreement.png',dpi=160)
            plt.close(fig)
        except Exception as plot_error:log_line('Plot error: '+repr(plot_error))
    manifest=[]
    for path in sorted(OUT.rglob('*')):
        if path.is_file() and path.name!='artifact_manifest.json':
            manifest.append(dict(path=str(path.relative_to(OUT)),bytes=path.stat().st_size,
                sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
    (OUT/'artifact_manifest.json').write_text(json.dumps(manifest,indent=2))
    archive_path=OUT.parent/(OUT.name+'.zip')
    with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as archive:
        for path in sorted(OUT.rglob('*')):
            if path.is_file():archive.write(path,arcname=str(path.relative_to(OUT)))
    ARCHIVE=str(archive_path)
    print('RESULT ARCHIVE',ARCHIVE,'size',archive_path.stat().st_size,flush=True)
    print('Contains checkpoints, training_steps.jsonl, training.log, cumulative_layer_results.csv, final_audit.json (if complete), and checksums.')
    if AUTO_DOWNLOAD_RESULTS and not RUN_OK:
        try:
            from google.colab import files
            files.download(ARCHIVE)
        except ImportError:
            from IPython.display import display,FileLink
            display(FileLink(ARCHIVE))
        except Exception as e:
            print('Browser blocked automatic download. Retrieve from Colab Files:',ARCHIVE,'reason',str(e))


## 7 · Fast native/student language evaluation (fixed)

Correctly handle both `Tensor` and `BatchEncoding` return formats from `apply_chat_template`. Measure gold-token NLL/accuracy and top-1 agreement on later validation windows, then run short uncached greedy prompts. Cached generation remains unsupported by the experimental adapters. These prompts are sanity checks, not a benchmark.

In [ ]:
# @title Fast native-versus-DeltaCeNN end-to-end evaluation
FAST_EVAL_RESULT={}
if RUN_FAST_EVAL:
    try:
        eval_offset=DEV_BLOCKS*SEQ_LEN
        EVAL=token_windows('validation',4,SEQ_LEN,start_token=eval_offset)
        with native_everywhere():
            native_metrics,native_top1=model_metrics(EVAL)
        student_metrics,_=model_metrics(EVAL,teacher_top1=native_top1)
        FAST_EVAL_RESULT=dict(native=native_metrics,student=student_metrics,
            delta_nll=student_metrics['nll']-native_metrics['nll'],
            installed_layers=installed_indices(),
            evaluation_source='WikiText-2 validation later windows; descriptive, not a blind external test')
        # Chat-template aware greedy comparison; no hybrid cache supported.
        FAST_CHAT_PROMPTS=[
            'What is 12 plus 19? Answer with the number.',
            'What is the capital of France? Answer in one sentence.',
        ]
        def slow_greedy(prompt,max_tokens=8):
            messages=[{'role':'user','content':prompt}]
            try:
                encoded=TOKENIZER.apply_chat_template(messages,tokenize=True,
                    add_generation_prompt=True,return_tensors='pt')
                # HF tokenizers may return Tensor OR BatchEncoding; both valid.
                ids=encoded['input_ids'] if isinstance(encoded,dict) else (
                    encoded.input_ids if hasattr(encoded,'input_ids') else encoded)
            except (TypeError,ValueError,KeyError,AttributeError):
                ids=TOKENIZER('User: '+prompt+'\nAssistant:',return_tensors='pt')['input_ids']
            if ids.ndim==1:ids=ids.unsqueeze(0)
            ids=ids.to(DEVICE)
            start=int(ids.shape[1]);ends=TOKENIZER.eos_token_id
            ends=set(ends if isinstance(ends,(list,tuple)) else [ends])
            im_end=TOKENIZER.convert_tokens_to_ids('<|im_end|>')
            if isinstance(im_end,int) and im_end>=0:ends.add(im_end)
            for _ in range(max_tokens):
                with torch.inference_mode(),torch.autocast('cuda',dtype=DTYPE):
                    logits=model(input_ids=ids,use_cache=False,logits_to_keep=1).logits[:,-1]
                token=logits.argmax(-1,keepdim=True)
                ids=torch.cat((ids,token),dim=-1)
                if int(token.item()) in ends:break
            return TOKENIZER.decode(ids[0,start:].detach().cpu(),skip_special_tokens=True)
        generations=[]
        for question in FAST_CHAT_PROMPTS:
            with native_everywhere(): teacher_text=slow_greedy(question,MAX_NEW_TOKENS)
            student_text=slow_greedy(question,MAX_NEW_TOKENS)
            result=dict(prompt=question,native=teacher_text,student=student_text,
                exact_text_agreement=(teacher_text.strip()==student_text.strip()))
            generations.append(result)
            print('\nPROMPT:',question,'\nNATIVE:',teacher_text,'\nSTUDENT:',student_text,flush=True)
        FAST_EVAL_RESULT['greedy_texts']=generations
        print('FAST GOLD EVAL:',json.dumps({k:v for k,v in FAST_EVAL_RESULT.items()
            if k!='greedy_texts'},indent=2))
    except BaseException as e:
        FAST_EVAL_RESULT={'error':repr(e),'installed_layers':installed_indices()}
        (OUT/'fast_eval_traceback.txt').write_text(traceback.format_exc())
        print('FAST EVAL FAILED:',repr(e))
    finally:
        (OUT/'fast_eval.json').write_text(json.dumps(FAST_EVAL_RESULT,indent=2,allow_nan=False))
else:
    print('Fast eval disabled by setting. Set RUN_FAST_EVAL=True to measure actual language modeling.')

## 8 · Package a **reloadable HF research adapter**, not a misleading base-model snapshot

The adapter bundle contains original Qwen model ID and pinned revision, all selected CeNN layer weights, an experiment manifest, the notebook's log/results, and a model card. It **does not** pretend that a plain `AutoModel.from_pretrained` call will recover nonstandard CeNN layers. Reload using [`load_delta_cenn_bundle`](https://github.com/vtavakkoli/TinyCeNN-LM/blob/main/src/tinycenn_lm/delta_cenn_v55_io.py). A forced or failed experiment is marked *research only*.

In [ ]:
# @title Export reloadable adapter package and Hugging Face model card
HF_BUNDLE=OUT/'hf_adapter_bundle'
SELECTED=installed_indices()
manifest=write_adapter_package(HF_BUNDLE,OUT,model_id=MODEL_ID,
    model_revision=REVISION,git_sha=GIT_SHA,
    selected_layers=SELECTED,full_layers=FULL_TARGETS,
    replacement_mode=REPLACEMENT_MODE,source_sha256=SOURCE_SHA256,
    run_fingerprint=RUN_FINGERPRINT,final_audit=FINAL,fast_eval=FAST_EVAL_RESULT,
    experimental=not bool(FINAL and FINAL.get('final_pass',False)))
card=prepare_hf_model_card(manifest,HF_REPO_ID)
(HF_BUNDLE/'README.md').write_text(card,encoding='utf-8')
print('Prepared reloadable adapter package:',HF_BUNDLE,'selected',SELECTED,
    'research_only',manifest['research_only'])

## 9 · Optional automatic Hugging Face upload

Default `HF_UPLOAD_MODE='research'` uploads **honestly labeled research checkpoints** if a writable `HF_TOKEN` is provided. Choose `quality_only` to upload only if the complete experiment passes validation and held-out gates; choose `disabled` for local-only runs. If no token is available, **skip safely** and keep the downloadable ZIP. HF secrets are not requested from the Colab vault by default, avoiding the earlier timeout warning. Set `HF_TOKEN` in the runtime environment, or enable `ENABLE_COLAB_SECRET_LOOKUP=True` after adding it in Colab Secrets with notebook access.

In [ ]:
# @title Publish adapters to Hugging Face when authorized
HF_STATUS=dict(mode=HF_UPLOAD_MODE,repo_id=HF_REPO_ID,uploaded=False)
try:
    if HF_UPLOAD_MODE not in ('disabled','quality_only','research'):
        raise ValueError('Unknown HF upload policy')
    allowed=HF_UPLOAD_MODE=='research' or (HF_UPLOAD_MODE=='quality_only' and
        bool(FINAL and FINAL.get('final_pass')) and len(SELECTED)>0)
    if HF_UPLOAD_MODE=='disabled':
        HF_STATUS['reason']='Disabled by experiment settings'
    elif not allowed:
        HF_STATUS['reason']='No validated deployable model: gated upload skipped'
    else:
        token=os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
        if not token and ENABLE_COLAB_SECRET_LOOKUP:
            try:
                from google.colab import userdata
                token=userdata.get('HF_TOKEN')
            except Exception as e:
                HF_STATUS['secret_lookup_error']=str(e)
        if not token:
            HF_STATUS['reason']='No writable HF_TOKEN in runtime; adapter is exported locally'
        else:
            from huggingface_hub import HfApi
            api=HfApi(token=token)
            api.create_repo(repo_id=HF_REPO_ID,repo_type='model',exist_ok=True)
            api.upload_folder(folder_path=str(HF_BUNDLE),repo_id=HF_REPO_ID,
                repo_type='model',commit_message='V5.5 CeNN research adapter with evaluation and provenance')
            HF_STATUS.update(uploaded=True,url='https://huggingface.co/'+HF_REPO_ID)
except Exception as e:
    HF_STATUS.update(uploaded=False,error=repr(e))
    print('HF upload unavailable, preserving local checkpoint:',repr(e))
finally:
    (OUT/'hf_upload_status.json').write_text(json.dumps(HF_STATUS,indent=2))
print('HF upload status:',json.dumps(HF_STATUS,indent=2))

## 10 · Final ZIP: training logs, per-layer checkpoints, real eval, HF adapter

The original progressive trainer already saved partial archives on errors. This final cell regenerates an archive including `fast_eval.json`, `hf_upload_status.json`, and the adapter folder; it requests a browser download at the end. If a browser blocks the automatic download, use the Colab Files sidebar.

In [ ]:
# @title Regenerate the complete research archive and download
for item in sorted(OUT.rglob('*')):
    if item.is_file() and item.name not in ('artifact_manifest.json',):
        pass
manifest_items=[]
for item in sorted(OUT.rglob('*')):
    if item.is_file() and item.name!='artifact_manifest.json':
        manifest_items.append(dict(path=str(item.relative_to(OUT)),bytes=item.stat().st_size,
            sha256=hashlib.sha256(item.read_bytes()).hexdigest()))
(OUT/'artifact_manifest.json').write_text(json.dumps(manifest_items,indent=2))
ARCHIVE=str(OUT.parent/(OUT.name+'.zip'))
with zipfile.ZipFile(ARCHIVE,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as zipout:
    for item in sorted(OUT.rglob('*')):
        if item.is_file():zipout.write(item,str(item.relative_to(OUT)))
print('COMPLETE DOWNLOAD:',ARCHIVE,'size',Path(ARCHIVE).stat().st_size)
if AUTO_DOWNLOAD_RESULTS:
    try:
        from google.colab import files
        files.download(ARCHIVE)
    except Exception as e:
        print('Browser download unavailable:',repr(e),'Use Colab Files sidebar:',ARCHIVE)
from IPython.display import display,FileLink
display(FileLink(ARCHIVE))

### Interpretation and deployment

- **Check `installed` in `final_audit.json`:** V5.4 installed 18 linear layers and **zero** full-attention layers; V5.5 similarly does not claim full-attention replacement merely because a candidate was trained.
- **Look for speedup >1.05×, validation agreement ≥0.98 and acceptable diagnostic NLL** before concluding that a deployment candidate is better.
- A chunked gated-delta operator plus a cellular state update is a *hybrid optimization*, not a stand-alone CeNN replacement of the entire recurrence.
- `fast_eval.json` includes native-versus-student text outputs, provided the GPU session reaches the evaluation cell.
- `hf_upload_status.json` distinguishes export from actual Hub upload; `quality_only` prevents publishing unsuccessful experiments as good models.